## 3.4.6 损失函数

### 1. 对数似然

softmax函数给出了一个向量$\hat{\mathbf{y}}$，
我们可以将其视为“对给定任意输入$\mathbf{x}$的每个类的条件概率”。
例如，$\hat{y}_1$=$P(y=\text{猫} \mid \mathbf{x})$。
假设整个数据集$\{\mathbf{X}, \mathbf{Y}\}$具有$n$个样本，
其中索引$i$的样本由特征向量$\mathbf{x}^{(i)}$和独热标签向量$\mathbf{y}^{(i)}$组成。
我们可以将估计值与实际值进行比较：

$$
P(\mathbf{Y} \mid \mathbf{X}) = \prod_{i=1}^n P(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)}).
$$

公式解释：

$P(\mathbf{Y} \mid \mathbf{X})$ 表示在已知整个输入数据集 $\mathbf{X}$ 的条件下，整个输出数据集 $\mathbf{Y}$ 出现的概率

$P(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)})$ 表示单个样本给定特征值x的条件下，输出为y的概率

$\prod_{i=1}^n$ 为连乘符号，表示从第1个元素连续相乘, 直到第n个元素

完整公式的意思是给定的所有输入数据集 $\mathbf{X}$, 产生输出数据集 $\mathbf{Y}$ 的概率等于给定每个独立样本生成独立样本输出的概率的连乘


根据最大似然估计，我们需要寻找一组参数，使得观测到的整个数据集输出为 $\mathbf{Y}$ 的概率最大, 我们要最大化$P(\mathbf{Y} \mid \mathbf{X})$

根据上述公式, 即需要最大化 $\prod_{i=1}^n P(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)})$

但是直接最大化一个连乘在数学上不太方便（乘积容易导致数值下溢，且求导复杂），同时根据对数函数是单调递增的特性，所以最大化原函数等价于最大化它的对数函数。

原函数是连乘，连乘函数的对数是对数的连加，因此可以转换为以下公式

$$
\log P(\mathbf{Y} \mid \mathbf{X}) = \log \prod_{i=1}^n P(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)})
= \sum_{i=1}^n \log P(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)})
$$

而在优化问题中，我们通常习惯最小化一个损失函数，所以我们可以转换为最小化负对数似然：

$$
-\log P(\mathbf{Y} \mid \mathbf{X}) = \sum_{i=1}^n -\log P(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)})
$$

现在问题变成：模型给出的 $P(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)})$ 具体是什么形式

对于分类问题，模型通常会输出一个q维的向量 $\hat{\mathbf{y}}^{(i)} = (\hat{y_1}, \hat{y_2},..., \hat{y_q})$

其中每个分量 $\hat{y_j}$ 表示模型预测样本属于第j类的概率，且满足 $\sum_{j=1}^q \hat{y_j} = 1$

而真实的标签 $\mathbf{y}_{(i)}$ 通常用 one-hot 向量表示：如果样本属于第 k 类，那么 $y_k = 1$, 其余分量都是0. 例如[0, 0, 1, 0, ..., 0]

那么在给定输入 $\mathbf{x}_{(i)}$ 的情况下，模型输出真实标签 $\mathbf{y}_{(i)}$ 的概率是多少？
由于 $\mathbf{y}_{(i)}$ 是一个one-hot向量，它只有一个分量为1，其余为0，所以这个概率就是模型预测的那个正确类别的概率值。即：

$$
P(\mathbf{y}_{(i)} \mid \mathbf{x}_{(i)}) = \prod_{j=1}^q(\hat{y}_j)^{y_j}
$$

代入负对数似然

$$
-\log P(\mathbf{y}^{(i)} \mid \mathbf{x}^{(i)}) = -\log (\prod_{j=1}^q(\hat{y}_j)^{y_j})
= -\sum_{j=1}^q \log ((\hat{y}_j)^{y_j}) = -\sum_{j=1}^q y_j \log \hat{y}_j
$$

可以推导出，对于任何标签$\mathbf{y}$和模型预测$\hat{\mathbf{y}}$，损失函数为：

$$ l(\mathbf{y}, \hat{\mathbf{y}}) = - \sum_{j=1}^q y_j \log \hat{y}_j. $$

### 2. softmax 及其导数

- softmax 函数：对于给定的logits向量 $\mathbf{o} = (o_1,o_2,...,o_q)$, softmax输出第 $j$ 个分量的概率为：

$$
\hat{y}_j = \operatorname*{softmax}(\mathbf{o})_j = \frac{\exp{(o_j)}}{\sum_{k=1}^q \exp{(o_k)}}
$$

- 交叉熵损失（单个样本）：

$$
l(\mathbf{y}, \mathbf{\hat{y}}) = -\sum_{j=1}^q y_j \log{\hat{y}_j}
$$

其中 $\mathbf{y}$ 是one-hot标签（只有一个 $y_j = 1$, 其余为0）

将 $\hat{y}_j$ 的表达式代入：

$$
l(\mathbf{y}, \mathbf{\hat{y}}) = -\sum_{j=1}^q y_j \log{\frac{\exp{(o_j)}}{\sum_{k=1}^q \exp{(o_k)}}}
$$

利用对数性质 $\log(a/b) = \log a - \log b$:

$$
\log({\frac{\exp{(o_j)}}{\sum_{k=1}^q \exp{(o_k)}}}) = \log(\exp{o_j}) - \log(\sum_{k=1}^q \exp(o_k))
= o_j - \log(\sum_{k=1}^q \exp(o_k))
$$

于是：

$$
l(\mathbf{y}, \mathbf{\hat{y}}) = -\sum_{j=1}^q y_j (o_j - \log(\sum_{k=1}^q \exp(o_k)))
= -\sum_{j=1}^q y_j o_j + \sum_{j=1}^q y_i \log(\sum_{k=1}^q \exp (o_k))
$$

第二项中 $\log (\sum_{k=1}^q \exp(o_k))$ 与 $j$ 无关，可以提到求和号之外：

$$
\sum_{j=1}^q y_i \log(\sum_{k=1}^q \exp (o_k)) = \log(\sum_{k=1}^q \exp(o_k)) \sum_{j=1}^q y_j
$$

由于 $\mathbf{y}$ 是 one-hot编码，所有 $y_i$ 之和为1，即 $\sum_{j=1}^q y_j = 1$，所以：

$$
\log(\sum_{k=1}^q \exp(o_k)) \sum_{j=1}^q y_j = \log(\sum_{k=1}^q \exp(o_k))
$$

因此推导出：

$$
l(\mathbf{y}, \mathbf{\hat{y}}) = -\sum_{j=1}^q y_j o_j + \log(\sum_{k=1}^q \exp(o_k))
= \log(\sum_{k=1}^q \exp(o_k)) - \sum_{j=1}^q y_j o_j
$$

对 $o_j$ 求偏导

现在将 $l$ 视为 $o_1,o_2,...o_q$ 的函数，求 $\frac{\partial{l}}{\partial{o_j}}$，为避免混淆，将第二项的求和下标改为 $k$

$$
l(\mathbf{y}, \mathbf{\hat{y}}) = \log(\sum_{k=1}^q \exp(o_k)) - \sum_{k=1}^q y_k o_k
$$

第一项的导数：

$$
\frac{\partial}{\partial_{o_j}} \log (\sum_{k=1}^q \exp(o_k)) = \frac{1}{\sum_{k=1}^q \exp(o_k)} \cdot \frac{\partial}{\partial_{o_j}} (\sum_{k=1}^q \exp(o_k))
= \frac{1}{\sum_{k=1}^q \exp(o_k)} \cdot \exp(o_j)
$$

第二项的导数：

$$
\frac{\partial}{\partial_{o_j}}(-\sum_{k=1}^q y_k o_k) = -y_j
$$

推导出：

$$
\frac{\partial{l}}{\partial_{o_j}} = \frac{\exp(o_j)}{\sum_{k=1}^q \exp(o_k)} - y_j = \operatorname*{softmax}(\mathbf{o})_j - y_j
$$